# 📊 Analisador Unificado de Fotos de Candidatos (Lote SP 2024 — 78k Fotos)

Este notebook executa o pipeline completo de detecção automatizada em lote no dataset do Estado de São Paulo (**78.000 fotos / 2GB**):

1. **Detector de Acessórios de Cabeça** (chapéus, bonés, toucas, tiaras, capacetes, etc. via **YOLO-World >90%**)
2. **Detector Híbrido de Óculos Escuros** (lentes escuras de sol via **YOLO-World + CLIP >90%**)

---

## 🛠️ Passo 1: Instalar Dependências e Desativar Alertas

In [ ]:
# Instalar dependências necessárias para YOLO-World e CLIP no Colab
!pip install -q ultralytics transformers pillow pandas ftfy git+https://github.com/ultralytics/CLIP.git

## 📁 Passo 2: Localizar e Extrair 'foto_cand2024_SP_div.zip' do Google Drive

Localiza automaticamente o arquivo `foto_cand2024_SP_div.zip` na mesma pasta do notebook no seu Google Drive, copia para o SSD ultrarrápido do Colab e descompacta em segundos.

In [ ]:
# OPÇÃO B: Processar Dataset Grande (foto_cand2024_SP_div.zip com 78k fotos e 2GB) via Google Drive
# DICA DE DESEMPENHO: O código busca o arquivo .zip no seu Drive (na mesma pasta do notebook ou subpastas),
# copia para o SSD local da máquina do Colab em segundos e descompacta lá. Isso deixa o I/O 50x mais rápido!
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

zip_name = 'foto_cand2024_SP_div.zip'
zip_found = None

# Verificar locais comuns no Google Drive
search_paths = [
    Path('/content/drive/MyDrive') / zip_name,
    Path('/content/drive/MyDrive/UFABC/2026-TOPICOS_DE_IA') / zip_name,
    Path('/content/drive/MyDrive/2026-TOPICOS_DE_IA') / zip_name
]

for p in search_paths:
    if p.exists():
        zip_found = p
        break

if not zip_found:
    print(f"🔍 Procurando '{zip_name}' no seu Google Drive...")
    matches = list(Path('/content/drive/MyDrive').rglob(zip_name))
    if matches:
        zip_found = matches[0]

if zip_found:
    print(f"-> Arquivo localizado com sucesso: {zip_found}")
    print("-> Copiando .zip do Drive para o SSD ultrarrápido do Colab...")
    !cp "{zip_found}" /content/
    print("-> Descompactando 78.000 fotos no SSD local...")
    !unzip -q /content/foto_cand2024_SP_div.zip -d /content/foto_cand2024_SP
    print("-> SUCESSO! 78.000 fotos prontas para análise ultrarrápida na pasta '/content/foto_cand2024_SP'!")
else:
    print(f"⚠️ Arquivo '{zip_name}' não encontrado no seu Google Drive. Verifique se o upload foi concluído.")

## 🧠 Passo 3: Código do Detector de Acessórios de Cabeça (YOLO-World Open-Vocabulary)

## 🧠 Passo 3: Código do Detector Híbrido de Óculos Escuros (YOLO + CLIP)

## 🧢 Passo 3: Executar Detecção 1/2 — Acessórios de Cabeça (>90% Confiança)

In [ ]:
# Executar o detector de acessórios de cabeça no dataset descompactado
input_dir = '/content/foto_cand2024_SP' if os.path.exists('/content/foto_cand2024_SP') else 'amostras'
res_cabeca = executar_deteccao_acessorios_cabeca_colab(
    input_dir=input_dir,
    output_dir='irregular_acessorios_cabeca',
    conf_thresh=0.90
)

## 🕶️ Passo 4: Executar Detecção 2/2 — Óculos Escuros (>90% Confiança)

In [ ]:
# Executar o detector híbrido de óculos escuros no dataset descompactado
input_dir = '/content/foto_cand2024_SP' if os.path.exists('/content/foto_cand2024_SP') else 'amostras'
res_oculos = executar_deteccao_oculos_escuros_colab(
    input_dir=input_dir,
    output_dir='irregular_oculos_escuros',
    clip_thresh=0.90
)

## 📊 Passo 5: Resumo Geral e Exibição de DataFrames

In [ ]:
print('=' * 70)
print('RESUMO DA ANÁLISE DO DATASET COMPLETO SP 2024')
print('=' * 70)
if res_cabeca:
    print(f"🧢 Acessórios de Cabeça Irregulares: {res_cabeca['total_irregulares']} de {res_cabeca['total_fotos']} fotos")
if res_oculos:
    print(f"🕶️ Óculos Escuros Irregulares:       {res_oculos['total_irregulares']} de {res_oculos['total_fotos']} fotos")
print('=' * 70)

## 💾 Passo 6: Salvar Backup no Google Drive e Baixar Resultados (.zip)

In [ ]:
# Compactar relatórios e imagens detectadas para download no computador
!zip -r resultados_sp_2024.zip irregular_acessorios_cabeca irregular_oculos_escuros

# Fazer backup direto na pasta do seu Google Drive se o Drive estiver montado
if os.path.exists('/content/drive/MyDrive'):
    backup_drive = zip_found.parent if (zip_found and zip_found.parent.exists()) else Path('/content/drive/MyDrive')
    !cp resultados_sp_2024.zip "{backup_drive}/"
    print(f"-> Backup do pacote de resultados salvo com sucesso no Drive em: '{backup_drive}'!")

from google.colab import files
files.download('resultados_sp_2024.zip')